# 03 · Busca por palavra-chave x busca semântica

Agora a parte divertida: **descobrir quem disse** a partir de uma ideia, e não da frase exata.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=UserWarning)   # aviso do pandas sobre conexão pyodbc
sys.path.append(str(pathlib.Path.cwd().parent))   # permite "from src import ..."
from src import config, banco
import time, json
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from src.vetores import IndiceEmMemoria

conn = banco.conectar(config.SQL_DATABASE)
modelo = SentenceTransformer(config.MODELO_EMBEDDING)
indice = IndiceEmMemoria.do_banco(conn, config.MODELO_EMBEDDING)
frases = pd.read_sql("SELECT * FROM rag.vw_Frases", conn).set_index("FraseId")
print("Índice em memória:", indice.matriz.shape, f"({indice.matriz.nbytes/1024:.0f} KB)")

## 1. Busca por palavra-chave (LIKE)

In [ ]:
def busca_palavra(termo):
    return pd.read_sql("EXEC rag.usp_BuscaPalavraChave @Termo = ?", conn, params=[termo])

busca_palavra("medo")

In [ ]:
busca_palavra("coragem")   # a ideia existe no banco, mas a palavra não

## 2. Busca semântica (cálculo no Python)

In [ ]:
def quem_disse(pergunta, k=3):
    vq = modelo.encode(pergunta, normalize_embeddings=True)
    res = indice.buscar(vq, k)
    out = frases.loc[[fid for fid, _ in res], ["Politico", "Texto", "Ano"]].copy()
    out.insert(0, "Similaridade", [round(s, 3) for _, s in res])
    return out

quem_disse("coragem para enfrentar o pânico")

## 3. Lado a lado: palavra-chave x significado

In [ ]:
perguntas = [
    "não desistir nunca da luta",
    "servir a pátria em vez de pedir coisas ao governo",
    "democracia é o povo governando",
    "cansado de ver a corrupção vencer",
    "derrubar a barreira que divide a cidade",
    "um país com dificuldade de manter a união",
]
for p in perguntas:
    kw = busca_palavra(p)
    top = quem_disse(p, 1).iloc[0]
    print(f"\n{p}")
    print(f"   LIKE: {len(kw)} resultado(s)")
    print(f"   Semântica: {top.Politico} ({top.Similaridade}) → {top.Texto[:70]}")

## 4. Busca entre idiomas
O modelo é multilíngue: perguntar em inglês ou espanhol encontra a frase gravada em português.

In [ ]:
quem_disse("we will never give up", 2)

In [ ]:
quem_disse("tenemos que derribar el muro", 2)

## 5. O "R" do RAG: montando o contexto para um LLM

RAG = **R**etrieval (buscar) + **A**ugmented (enriquecer o prompt) + **G**eneration (o LLM responde). O SQL Server 2017 + Python resolvem o **R**. O prompt abaixo pode ser enviado para qualquer LLM.

In [ ]:
def montar_prompt(pergunta, k=3):
    res = quem_disse(pergunta, k)
    contexto = "\n".join(f"- \"{r.Texto}\" ({r.Politico}, {r.Ano})" for r in res.itertuples())
    return f"""Responda usando APENAS as frases abaixo. Se não houver resposta, diga que não sabe.

Frases recuperadas do SQL Server:
{contexto}

Pergunta: {pergunta}"""

print(montar_prompt("Quem falou sobre não ter medo?"))

## 6. Bônus para DBAs: cosseno em T-SQL puro no 2017

Sem `VECTOR` e sem `VECTOR_DISTANCE`, mas com `OPENJSON` (disponível desde o 2016) e um `SUM` de produtos. Funciona, e é ótimo para entender a matemática. Na prática, para volume, a aplicação é mais rápida.

In [ ]:
pergunta = "coragem para enfrentar o pânico"
vq = modelo.encode(pergunta, normalize_embeddings=True)
vetor_json = json.dumps([round(float(x), 7) for x in vq])

t = time.perf_counter()
tsql = pd.read_sql("EXEC rag.usp_BuscaSemanticaTSQL @VetorJson = ?, @Modelo = ?, @K = 3",
                   conn, params=[vetor_json, config.MODELO_EMBEDDING])
print(f"T-SQL: {(time.perf_counter()-t)*1000:.1f} ms")
tsql

O resultado bate com o do Python? Deveria: é a mesma conta, feita em lugares diferentes.

In [ ]:
quem_disse(pergunta, 3)

## 7. E se forem 1 milhão de frases?

Simulação com vetores aleatórios para medir a força bruta em NumPy.

In [ ]:
n, d = 1_000_000, 384
mat = np.random.randn(n, d).astype("float32")
mat /= np.linalg.norm(mat, axis=1, keepdims=True)
q = mat[0]
t = time.perf_counter()
top = np.argsort(-(mat @ q))[:5]
print(f"{n:,} vetores | {mat.nbytes/1024**3:.2f} GB em RAM | busca em {(time.perf_counter()-t)*1000:.0f} ms")
del mat

**Leitura do resultado:**
- Até centenas de milhares de vetores, força bruta em memória resolve.
- Milhões de vetores: use um índice ANN em memória (FAISS, hnswlib) carregado a partir do SQL Server.
- O SQL Server continua sendo a **fonte da verdade**: backup, segurança, transação e governança.
- Quando migrar para o 2025, os bytes viram `VECTOR(384)` e a conta vai para `VECTOR_DISTANCE`.